### Clustering (DBSCAN)

In [1]:
import numpy as np
import pandas as pd
import optuna
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_score

In [3]:
# Load your data
lrt_house_pca = pd.read_csv('lrt_house_pca.csv')
lrt_condo_pca = pd.read_csv('lrt_condo_pca.csv')
lrt_apt_pca = pd.read_csv('lrt_apt_pca.csv')
lrt_com_pca = pd.read_csv('lrt_com_pca.csv')
lrt_land_pca = pd.read_csv('lrt_land_pca.csv')

In [5]:
lrt_house_pca = lrt_house_pca.drop_duplicates()
lrt_condo_pca = lrt_condo_pca.drop_duplicates()
lrt_apt_pca = lrt_apt_pca.drop_duplicates()
lrt_com_pca = lrt_com_pca.drop_duplicates()
lrt_land_pca = lrt_land_pca.drop_duplicates()

In [7]:
X_house = lrt_house_pca[['LRTHubDist', 'PC1', 'PC2', 'PC3', 'PC4']]
X_condo = lrt_condo_pca[['LRTHubDist', 'PC1', 'PC2', 'PC3', 'PC4']]
X_apt = lrt_apt_pca[['LRTHubDist', 'PC1', 'PC2', 'PC3', 'PC4']]
X_com = lrt_com_pca[['LRTHubDist', 'PC1', 'PC2', 'PC3', 'PC4']]
X_land = lrt_land_pca[['LRTHubDist', 'PC1', 'PC2', 'PC3', 'PC4']]

In [9]:
def objective(trial, X):
    """
    Objective function for Optuna to optimize DBSCAN hyperparameters using silhouette score.
    
    Parameters:
        trial: Optuna trial object
        X: Feature matrix
    
    Returns:
        Silhouette score (higher is better)
    """
    # Suggest hyperparameters
    eps = trial.suggest_float("eps", 0.1, 5.0, log=True)  # Search range for neighborhood radius
    min_samples = trial.suggest_int("min_samples", 2, 20)  # Minimum points in a cluster
    
    # Create DBSCAN pipeline
    dbscan_pipeline = make_pipeline(StandardScaler(), DBSCAN(eps=eps, min_samples=min_samples))
    
    # Fit DBSCAN model
    labels = dbscan_pipeline.fit_predict(X)
    
    # Silhouette score requires at least 2 clusters
    num_clusters = len(set(labels) - {-1})  # Exclude noise (-1)
    if num_clusters < 2:
        return -1  # Return a bad score if only one cluster is found

    # Compute silhouette score
    score = silhouette_score(X, labels)
    return score

def tune_dbscan(X, n_trials=50):
    """
    Uses Optuna to find the best DBSCAN hyperparameters.
    
    Parameters:
        X: Feature matrix
        n_trials: Number of optimization trials (default: 50)
    
    Returns:
        Best parameters and corresponding DBSCAN pipeline
    """
    study = optuna.create_study(direction="maximize")  # Maximize silhouette score
    study.optimize(lambda trial: objective(trial, X), n_trials=n_trials)
    
    best_params = study.best_params
    print("Best Hyperparameters:", best_params)
    
    # Train final DBSCAN model with best parameters
    best_dbscan_pipeline = make_pipeline(StandardScaler(), DBSCAN(**best_params))
    best_dbscan_pipeline.fit(X)

    return best_dbscan_pipeline, best_params

### House

In [12]:
# Example usage:
best_dbscan, best_dbscan_params = tune_dbscan(X_house)

[I 2025-03-21 21:51:33,539] A new study created in memory with name: no-name-1cd28ae5-4403-4bb0-8a51-58e97781a7c0
[I 2025-03-21 21:51:37,712] Trial 0 finished with value: 0.09768687815729091 and parameters: {'eps': 0.5095782954376638, 'min_samples': 5}. Best is trial 0 with value: 0.09768687815729091.
[I 2025-03-21 21:51:42,583] Trial 1 finished with value: 0.06304268859290926 and parameters: {'eps': 0.9929061380043867, 'min_samples': 3}. Best is trial 0 with value: 0.09768687815729091.
[I 2025-03-21 21:51:46,436] Trial 2 finished with value: -0.16778989460127808 and parameters: {'eps': 0.22085337403215205, 'min_samples': 17}. Best is trial 0 with value: 0.09768687815729091.
[I 2025-03-21 21:51:50,404] Trial 3 finished with value: -0.39092484340461214 and parameters: {'eps': 0.10692155178945832, 'min_samples': 19}. Best is trial 0 with value: 0.09768687815729091.
[I 2025-03-21 21:51:54,350] Trial 4 finished with value: -0.0035109217446136413 and parameters: {'eps': 0.33901499760376225,

Best Hyperparameters: {'eps': 1.1338523544030148, 'min_samples': 9}


In [14]:
def get_top_features_and_silhouette(X, best_dbscan):
    """
    Extracts the top 5 distinguishing features for each cluster 
    and computes the silhouette score.
    
    Parameters:
        X: Feature matrix (DataFrame or NumPy array)
        best_dbscan: Trained DBSCAN model

    Returns:
        top_features_df: DataFrame of top 5 features per cluster
        best_silhouette: Silhouette score of the best clustering
    """
    # Get cluster labels
    labels = best_dbscan.fit_predict(X)

    # Compute silhouette score (only if there are at least 2 clusters)
    num_clusters = len(set(labels) - {-1})  # Exclude noise (-1)
    best_silhouette = silhouette_score(X, labels) if num_clusters > 1 else -1

    # Convert to DataFrame if needed and append cluster labels
    X_df = pd.DataFrame(X) if not isinstance(X, pd.DataFrame) else X.copy()
    X_df["Cluster"] = labels  
    clustered_data = X_df[X_df["Cluster"] != -1]  # Remove noise

    top_features_dict = {}

    for cluster in np.unique(clustered_data["Cluster"]):
        cluster_data = clustered_data[clustered_data["Cluster"] == cluster].drop(columns=["Cluster"])
        
        feature_means = cluster_data.mean()
        top_features = feature_means.abs().nlargest(5).index.tolist()
        top_values = feature_means[top_features].values.tolist()

        top_features_dict[f"Cluster {cluster}"] = list(zip(top_features, top_values))

    return pd.DataFrame(top_features_dict), best_silhouette

In [16]:
# Example Usage
top_features_dbscan, best_silhouette = get_top_features_and_silhouette(X_house, best_dbscan)

print(f"Best Silhouette Score: {best_silhouette}")
print(top_features_dbscan)

Best Silhouette Score: 0.19948360343283508
                          Cluster 0                           Cluster 1
0         (PC1, 0.1974945532115431)           (PC3, 3.9822155049224333)
1       (PC2, -0.13894433238324852)             (PC1, 3.05947457763294)
2  (LRTHubDist, 0.0727218922266301)           (PC4, 2.7539908055813167)
3       (PC3, -0.06822210577333156)         (PC2, -0.37096714734309355)
4       (PC4, 0.005604854061705756)  (LRTHubDist, 0.028052865554995435)


In [18]:
# Save results into a copy of X_house
X_dbscan = lrt_house_pca.copy()
X_dbscan["DBSCAN_Cluster"] = best_dbscan.fit_predict(X_house)

# Save as CSV
X_dbscan.to_csv("house_pca_tuned_dbscan.csv", index=False)

### Condo

In [20]:
# Example usage:
best_dbscan, best_dbscan_params = tune_dbscan(X_condo)

[I 2025-03-21 21:56:03,825] A new study created in memory with name: no-name-e8eb722a-2c4d-4d94-b419-5550213293bd
[I 2025-03-21 21:56:06,714] Trial 0 finished with value: -1.0 and parameters: {'eps': 3.759948382772783, 'min_samples': 2}. Best is trial 0 with value: -1.0.
[I 2025-03-21 21:56:09,864] Trial 1 finished with value: -0.02639267111607585 and parameters: {'eps': 0.28429826473616193, 'min_samples': 16}. Best is trial 1 with value: -0.02639267111607585.
[I 2025-03-21 21:56:13,556] Trial 2 finished with value: 0.0597544641601152 and parameters: {'eps': 0.4576718756821525, 'min_samples': 10}. Best is trial 2 with value: 0.0597544641601152.
[I 2025-03-21 21:56:17,632] Trial 3 finished with value: 0.42835837114233466 and parameters: {'eps': 1.3452189856438466, 'min_samples': 18}. Best is trial 3 with value: 0.42835837114233466.
[I 2025-03-21 21:56:20,499] Trial 4 finished with value: -0.04204627111916891 and parameters: {'eps': 0.1149102611832151, 'min_samples': 7}. Best is trial 3 

Best Hyperparameters: {'eps': 1.2542453815523624, 'min_samples': 17}


In [22]:
# Example Usage
top_features_dbscan, best_silhouette = get_top_features_and_silhouette(X_condo, best_dbscan)

print(f"Best Silhouette Score: {best_silhouette}")
print(top_features_dbscan)

Best Silhouette Score: 0.44439577044146855
                           Cluster 0                          Cluster 1  \
0          (PC1, 2.8763495746271337)          (PC3, 3.7768903786573924)   
1  (LRTHubDist, -0.9519170532196178)          (PC1, 1.9643936781362419)   
2         (PC4, -0.2038331123198522)           (PC4, 1.555178790951632)   
3       (PC2, -0.025392483302644975)         (PC2, -1.2270169102761235)   
4       (PC3, -0.019410555431991078)  (LRTHubDist, -0.8455717340750872)   

                          Cluster 2  
0        (PC1, -2.6144562540360403)  
1  (LRTHubDist, 0.8767313476376973)  
2        (PC4, 0.19482096008323985)  
3        (PC2, 0.08121359268548728)  
4       (PC3, -0.06695413037900891)  


In [24]:
# Save results into a copy of X_house
X_dbscan = lrt_condo_pca.copy()
X_dbscan["DBSCAN_Cluster"] = best_dbscan.fit_predict(X_condo)

# Save as CSV
X_dbscan.to_csv("condo_pca_tuned_dbscan.csv", index=False)

### Apt

In [26]:
# Example usage:
best_dbscan, best_dbscan_params = tune_dbscan(X_apt)

[I 2025-03-21 21:59:59,173] A new study created in memory with name: no-name-e377573d-d359-4244-b16f-89c9f99d7477
[I 2025-03-21 21:59:59,191] Trial 0 finished with value: -1.0 and parameters: {'eps': 4.431658442207412, 'min_samples': 15}. Best is trial 0 with value: -1.0.
[I 2025-03-21 21:59:59,221] Trial 1 finished with value: -1.0 and parameters: {'eps': 1.106036967595834, 'min_samples': 10}. Best is trial 0 with value: -1.0.
[I 2025-03-21 21:59:59,242] Trial 2 finished with value: -1.0 and parameters: {'eps': 0.31683998455865453, 'min_samples': 17}. Best is trial 0 with value: -1.0.
[I 2025-03-21 21:59:59,266] Trial 3 finished with value: -1.0 and parameters: {'eps': 2.458783091871694, 'min_samples': 8}. Best is trial 0 with value: -1.0.
[I 2025-03-21 21:59:59,282] Trial 4 finished with value: -1.0 and parameters: {'eps': 0.2814346232310221, 'min_samples': 13}. Best is trial 0 with value: -1.0.
[I 2025-03-21 21:59:59,306] Trial 5 finished with value: -1.0 and parameters: {'eps': 0.2

Best Hyperparameters: {'eps': 1.6156042503115806, 'min_samples': 11}


In [54]:
# Example Usage
top_features_dbscan, best_silhouette = get_top_features_and_silhouette(X_apt, best_dbscan)

print(f"Best Silhouette Score: {best_silhouette}")
print(top_features_dbscan)

Best Silhouette Score: 0.293389217521979
                            Cluster 0                          Cluster 1  \
0           (PC1, 1.4176362174505972)          (PC2, 1.5989679570872797)   
1          (PC2, -0.7401035226231806)         (PC4, -1.1598927766952623)   
2         (PC3, -0.22547612196596742)         (PC3, -0.6149661394926974)   
3          (PC4, 0.04310600039782284)         (PC1, -0.2924228412379172)   
4  (LRTHubDist, 0.013081155468962559)  (LRTHubDist, -0.1827269067777143)   

                           Cluster 2                          Cluster 3  \
0         (PC3, -2.7117336830250323)         (PC1, -2.3864179653952475)   
1          (PC2, 2.6178124692503983)         (PC2, -1.4645114867463138)   
2         (PC1, -1.2481481886727288)          (PC4, 1.2768076660849637)   
3  (LRTHubDist, -0.6215117421070613)           (PC3, 0.552955455586916)   
4         (PC4, -0.0283660764144383)  (LRTHubDist, -0.5314447228627811)   

                            Cluster 4              

In [56]:
# Save results into a copy of X_house
X_dbscan = lrt_apt_pca.copy()
X_dbscan["DBSCAN_Cluster"] = best_dbscan.fit_predict(X_apt)

# Save as CSV
X_dbscan.to_csv("apt_pca_tuned_dbscan.csv", index=False)

### Commercial Real Estate

In [58]:
# Example usage:
best_dbscan, best_dbscan_params = tune_dbscan(X_com)

[I 2025-03-20 22:45:08,408] A new study created in memory with name: no-name-7d3f6d6f-654a-4f23-bce3-b5ce7752c638
[I 2025-03-20 22:45:08,648] Trial 0 finished with value: 0.05308655013926951 and parameters: {'eps': 0.1522259761275263, 'min_samples': 20}. Best is trial 0 with value: 0.05308655013926951.
[I 2025-03-20 22:45:08,837] Trial 1 finished with value: -1.0 and parameters: {'eps': 1.8935618609390235, 'min_samples': 9}. Best is trial 0 with value: 0.05308655013926951.
[I 2025-03-20 22:45:09,052] Trial 2 finished with value: 0.36243481998456634 and parameters: {'eps': 0.2239463046663967, 'min_samples': 7}. Best is trial 2 with value: 0.36243481998456634.
[I 2025-03-20 22:45:09,251] Trial 3 finished with value: -1.0 and parameters: {'eps': 4.575135082522199, 'min_samples': 5}. Best is trial 2 with value: 0.36243481998456634.
[I 2025-03-20 22:45:09,513] Trial 4 finished with value: 0.19493564114177397 and parameters: {'eps': 0.6373235077390349, 'min_samples': 2}. Best is trial 2 with

Best Hyperparameters: {'eps': 4.602190542874389, 'min_samples': 2}


In [28]:
# Example Usage
top_features_dbscan, best_silhouette = get_top_features_and_silhouette(X_com, best_dbscan)

print(f"Best Silhouette Score: {best_silhouette}")
print(top_features_dbscan)

Best Silhouette Score: 0.38599999134912033
                           Cluster 0                       Cluster 1
0           (PC1, 2.126921207281861)      (PC1, -1.7009956814428875)
1  (LRTHubDist, -0.8107943022836188)  (LRTHubDist, 0.69834665723461)
2          (PC2, 0.2577504307021628)      (PC2, -0.5379023577296841)
3        (PC3, -0.23685301408569034)       (PC3, 0.3836861665038819)
4        (PC4, -0.10654499582586147)     (PC4, 0.029212150216347793)


In [30]:
# Save results into a copy of X_house
X_dbscan = lrt_com_pca.copy()
X_dbscan["DBSCAN_Cluster"] = best_dbscan.fit_predict(X_com)

# Save as CSV
X_dbscan.to_csv("com_pca_tuned_dbscan.csv", index=False)

### Land

In [32]:
# Example usage:
best_dbscan, best_dbscan_params = tune_dbscan(X_land)

[I 2025-03-21 22:00:56,640] A new study created in memory with name: no-name-40bc9476-c727-4994-8be8-a34edf971e63
[I 2025-03-21 22:00:57,029] Trial 0 finished with value: 0.08438969819062686 and parameters: {'eps': 0.37032643834252965, 'min_samples': 16}. Best is trial 0 with value: 0.08438969819062686.
[I 2025-03-21 22:00:57,395] Trial 1 finished with value: 0.23812928183078202 and parameters: {'eps': 0.6067539431515767, 'min_samples': 10}. Best is trial 1 with value: 0.23812928183078202.
[I 2025-03-21 22:00:57,659] Trial 2 finished with value: -1.0 and parameters: {'eps': 1.9609800438533005, 'min_samples': 9}. Best is trial 1 with value: 0.23812928183078202.
[I 2025-03-21 22:00:58,055] Trial 3 finished with value: -1.0 and parameters: {'eps': 2.9819198371980433, 'min_samples': 20}. Best is trial 1 with value: 0.23812928183078202.
[I 2025-03-21 22:00:58,455] Trial 4 finished with value: 0.1760910527753104 and parameters: {'eps': 0.6160601381392911, 'min_samples': 11}. Best is trial 1 

Best Hyperparameters: {'eps': 0.3014758694221961, 'min_samples': 2}


In [34]:
# Example Usage
top_features_dbscan, best_silhouette = get_top_features_and_silhouette(X_land, best_dbscan)

print(f"Best Silhouette Score: {best_silhouette}")
print(top_features_dbscan)

Best Silhouette Score: 0.3834050726371641
                           Cluster 0                         Cluster 1  \
0          (PC3, 3.2764423576639374)  (LRTHubDist, -1.244985001178767)   
1          (PC4, 1.5973159891298543)        (PC1, -0.9847606493144316)   
2  (LRTHubDist, -1.1434697351833132)         (PC3, 0.9597090212355615)   
3          (PC2, -0.689534658254917)        (PC4, -0.7879446060174137)   
4         (PC1, 0.31179886249965555)        (PC2, -0.2131511750462428)   

                           Cluster 2                           Cluster 3  \
0         (PC2, -1.0561691107218751)            (PC2, -1.26673492636522)   
1  (LRTHubDist, -0.7124130916209619)          (PC1, -1.1584720214063209)   
2         (PC1, -0.6283529204714007)          (PC3, -0.5533154235782738)   
3          (PC4, 0.4873981272874722)  (LRTHubDist, -0.13176250345619153)   
4         (PC3, 0.18085429863385483)          (PC4, 0.03954453754581338)   

                           Cluster 4                    

In [36]:
# Save results into a copy of X_house
X_dbscan = lrt_land_pca.copy()
X_dbscan["DBSCAN_Cluster"] = best_dbscan.fit_predict(X_land)

# Save as CSV
X_dbscan.to_csv("land_pca_tuned_dbscan.csv", index=False)

### Further Clustering (DBSCAN)

House: Tuned DBSCAN

Condo: Tuned DBSCAN

Apt: BIC

H
Commercial: Tuned DSC

AN
Land: BIRCH 

House (4 clusters, -1, 0, 1, 2): Remove clusters -1, 1, 2

Condo (4 clusters, -1, 0, 1, 2): Remove clusters -1, 1, 2

Apt (4 clusters, 0, 1, 2, 3): Remove clusters 1, 2, 3

Com (3 clusters, -1, 0, 1): Remove clusters -1, 1

Land (5 clusters, 0, 1, 2, 3, 4): Remove clusters 0, 2, 3, 4